# Model Monitoring & Data Drift (Evidently AI)

In [ ]:
# Day 38 — Model Monitoring & Data Drift
# Deploying an ML model is Day 1. The real work starts on Day 2:
# Models silently degrade in production because the world changes!
#
# Three types of drift:
#   1. Data Drift       -> Input features shift (e.g. traffic spikes, new app version)
#   2. Concept Drift    -> Relationship between X and y changes
#   3. Prediction Drift -> Model output distribution shifts (e.g. 90% 'critical' suddenly)
#
# Evidently AI is the open-source Python standard for ML monitoring.
#
# Today we'll cover:
#   1. Reference (training) vs Current (production) datasets
#   2. Evidently Report with DataDriftPreset
#   3. Programmatic drift checks (JSON output for automated alerts)
#   4. Triggering retraining when drift exceeds a threshold
#
# Install: pip install evidently

### 1. Simulating Reference vs Production Drift Data

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Reference data (what the model was trained on)
reference_df = pd.DataFrame({
    "cpu": np.random.normal(50, 12, 300),
    "memory": np.random.normal(55, 10, 300),
    "error_count": np.random.poisson(3, 300)
})

# Current production data (a memory leak + traffic surge shifted CPU & memory!)
current_df = pd.DataFrame({
    "cpu": np.random.normal(78, 15, 300),        # drifted!
    "memory": np.random.normal(82, 8, 300),      # drifted!
    "error_count": np.random.poisson(3, 300)     # stable
})

print("Reference CPU mean:", round(reference_df["cpu"].mean(), 2))
print("Current CPU mean:  ", round(current_df["cpu"].mean(), 2))

### 2. Running Evidently DataDriftPreset

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset

# Create and run the drift report
drift_report = Report(metrics=[
    DataDriftPreset(),
    DataQualityPreset()
])

drift_report.run(reference_data=reference_df, current_data=current_df)

# Save interactive HTML report (open in browser or attach as MLflow artifact!)
drift_report.save_html("drift_report.html")
print("Saved interactive report to drift_report.html")

### 3. Automated Drift Alerts in Python Pipelines

In [ ]:
# In an automated MLOps pipeline (Prefect/Airflow), we don't read HTML.
# We extract the dictionary/JSON result and check boolean flags!

result = drift_report.as_dict()

dataset_drift = result["metrics"][0]["result"]["dataset_drift"]
drift_share = result["metrics"][0]["result"]["share_of_drifted_columns"]

print(f"Dataset Drift Detected: {dataset_drift}")
print(f"Share of Drifted Columns: {drift_share:.0%}")

if dataset_drift or drift_share >= 0.5:
    print("ALERT: Significant data drift detected -> Triggering retraining pipeline!")
else:
    print("OK: Data distribution is stable.")

In [ ]:
# Practice Questions:
#
# 1. What is the difference between Data Drift and Concept Drift?
# 2. Why is `as_dict()` essential for CI/CD or scheduled monitoring jobs?
# 3. How would you log `drift_report.html` into MLflow (Day 34)?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Production Drift Detector
# Write a function `check_drift(ref_df, prod_df, threshold=0.3)` that returns
# a dict with `drift_detected` (bool), `drifted_columns` (list), and `drift_share` (float).

# Exercise 2: Model Accuracy Decay Alerter
# Simulate ground-truth labels arriving 24h later and use `ClassificationPreset()`
# to detect if F1 score drops below 0.85.

# Exercise 3: Automated Retraining Trigger
# Combine Pandera (Day 36) + Evidently (Day 38): first validate schema,
# then check drift, and only trigger retraining if data is valid AND drifted.